# **Stanley-Reisner Attention Analysis**

```
Attention weights → Threshold → Simplicial complex Δ
                                    ↓
                         Stanley-Reisner ideal I_Δ
                                    ↓
                         Minimal free resolution → Betti table
                                    ↓
                         Hochster's formula → Topological invariants

In [ ]:
!git clone https://git.hodgederrick.com/hodge360/stanley-reisner-attention

In [ ]:
!cat stanley-reisner-attention/sr_attention/pyproject.toml

In [ ]:
# 1. Fix the invalid build-backend in pyproject.toml
!sed -i 's/setuptools.backends.legacy:build/setuptools.build_meta/' stanley-reisner-attention/sr_attention/pyproject.toml

# 2. Install a version of setuptools that satisfies both the project (>=68) and torch (<82)
!pip install "setuptools>=68,<82"

# 3. Install the package in editable mode
!cd stanley-reisner-attention/sr_attention && pip install -e ."[dev]" --no-build-isolation

In [ ]:
import sys
import os

# Add the repository root to sys.path so 'import sr_attention' works internally
repo_root = os.path.abspath('stanley-reisner-attention')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)


In [ ]:
from sr_attention.extraction import AttentionExtractor
from sr_attention.complex_builder import AttentionComplexBuilder
from sr_attention.sr_core import stanley_reisner_generators, primary_decomposition_from_facets
from sr_attention.persistence import betti_table_via_hochster, format_betti_table

# 1. Load model and extract attention patterns
extractor = AttentionExtractor("Qwen/Qwen2.5-0.5B")
cache = extractor.run("The mathematician proved the theorem using algebra.")
print(f"Sequence Length: {cache.seq_len}")

# 2. Build the filtered simplicial complex for layer 0, head 0
attn = cache.head(layer=8, head=5)
builder = AttentionComplexBuilder()
st = builder.build(attn, token_labels=cache.tokens)

# 3. Extract SR ideal generators
gens, monomials = stanley_reisner_generators(st, token_labels=cache.tokens, verbose=True)

# 4. Betti table (Topological analysis)
betti = betti_table_via_hochster(st)
print("\nBetti Table:")
print(format_betti_table(betti))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
Sequence Length: 10
SR ideal I(Δ) at τ=0.010 (t=0.990): 53 generators
  The· the
  The· theorem
  The· algebra
   mathematic· the
   mathematic· theorem
   mathematic· using
   mathematic· algebra
  ian· theorem
  ian· algebra
   proved· algebra
   the· algebra
   theorem·.
  <|endoftext|>·The· mathematic·ian
  <|endoftext|>·The· mathematic· proved
  <|endoftext|>·The· mathematic·.
  … and 38 more

Betti Table:
i\j	0	1	2	3	4	5	6	7	8
0	1	0	0	0	0	0	0	0	0
1	0	0	0	0	0	0	0	0	0
2	0	0	0	23	0	0	0	0	0
3	0	0	0	0	24	5	0	0	0
4	0	0	0	0	0	9	9	0	0
5	0	0	0	0	0	0	1	5	0
6	0	0	0	0	0	0	0	0	1


In [ ]:
# Next code cell: Extract facets and verify the "two disjoint cliques" hypothesis
import gudhi

# Extract all simplices and their filtration values
simplices = list(st.get_simplices())

# Get the dimension of the complex
max_dim = st.dimension()
print(f"Complex dimension: {max_dim}")

# Fix: Calculate number of simplices per dimension manually
dim_counts = {d: 0 for d in range(max_dim + 1)}
for simplex, filtration in simplices:
    dim = len(simplex) - 1
    dim_counts[dim] += 1

for d, count in dim_counts.items():
    print(f"Simplices of dimension {d}: {count}")

# Total simplices
total = st.num_simplices()
print(f"Total simplices: {total}")

# Get all vertices
vertices = list(st.get_skeleton(0))
print(f"\nVertices: {len(vertices)}")

# Get the 1-skeleton (edges) to see the graph structure
edges = list(st.get_skeleton(1))
# Show edges with non-zero filtration (i.e., present in the complex)
edges_present = [(e, f) for e, f in edges if len(e) == 2]
print(f"Present edges: {len(edges_present)}")
for e, f in edges_present[:20]:
    print(f"  {e} -> filtration {f}")

Complex dimension: 1
Simplices of dimension 0: 10
Simplices of dimension 1: 1
Total simplices: 11

Vertices: 10
Present edges: 1
  [0, 1] -> filtration 0.6531727313995361


# NOTES



### 1. Simplicial Complex Δ → Face Ring S/I_Δ

**Correct.** This is the definition of the Stanley-Reisner construction. No gaps here.

**But:** The ring S/I_Δ is **not** a geometric object in the usual sense. It's a combinatorial commutative algebra object. To get geometry, you need additional structure (a grading, a toric embedding, etc.).

---

### 2. S/I_Δ → Minimal Free Resolution → Betti Table

**Correct.** The Betti numbers β_{i,j} = dim_k Tor_i^S(S/I_Δ, k)_j are computed via the minimal free resolution.

**But:** The Betti table is an **algebraic invariant**, not a topological one. Hochster's formula relates it to **homology of subcomplexes**, not directly to the homotopy type of Δ.

**Gap I glossed over:** Two complexes can have the same Betti table but different homotopy types. The Betti table is **not a complete homotopy invariant**. It's complete for certain classes (e.g., Gorenstein* complexes), but not in general.

---

### 3. Betti Table → Homotopy Type

**Overstated.** I said "the Betti signature is often rigid enough to identify the homotopy type up to wedge of spheres."

**Correction:** The Betti table determines the **Hilbert series** of the Tor modules, which by Hochster's formula is a weighted sum of **reduced homology dimensions** of induced subcomplexes. It does **not** determine:

- The **cup product structure** in cohomology
- The **Massey products** (higher-order cohomology operations)
- The **homotopy groups** π_i(Δ) for i > 1
- Whether Δ is a **wedge of spheres** or has more complicated attaching maps

**What it does give you:** Constraints. If β_{i,j} ≠ 0, then some induced subcomplex has non-trivial homology. But the converse is not a homotopy classification.

---

### 4. Homotopy Type → Sheaf Cohomology

**Gap I jumped over.** I went from "homotopy type" to "sheaf cohomology on the face poset" without explaining the intermediate step.

**The actual connection:**

The face poset P(Δ) has **order topology** (Alexandrov topology). Sheaves on P(Δ) are equivalent to **functors** from P(Δ)^op to Vect_k. The sheaf cohomology H^i(P(Δ); ℱ) can be computed via the **order complex** of P(Δ), which is the **barycentric subdivision** of Δ.

**Key point:** The sheaf cohomology of the constant sheaf on P(Δ) is **isomorphic to the simplicial cohomology of Δ** (via barycentric subdivision). So for the constant sheaf, there's no new information.

**Where sheaf cohomology adds value:** For **non-constant sheaves** — constructible sheaves, local systems, sheaves of modules. These can distinguish spaces that ordinary cohomology cannot.

---

### 5. Sheaf Cohomology → "Information Flow"

**Speculative / overstated.** I said: "The sheaf restriction maps are the formalization of attention as information transfer."

**Correction:** This is an **analogy**, not a theorem. The mathematical connection is:

- A sheaf on P(Δ) assigns data to each face and restriction maps to inclusions
- Attention assigns weights to token pairs and propagates through the network

The **formal similarity** is clear: both are "local-to-global" structures with consistency conditions. But the **actual mapping** from attention weights to sheaf data requires:

1. A choice of **stalk functor** (what vector space to assign to each face)
2. A choice of **restriction maps** (how to map between them)
3. A proof that the **gluing axioms** correspond to attention dynamics

**None of this is standard.** It's a research direction, not an established tool.

---

### 6. Deformation Theory → Threshold Stability

**Speculative / possibly wrong.** I said: "Deformations of S/I_Δ correspond to perturbations of the attention threshold."

**Correction:** The deformation theory of Stanley-Reisner rings is well-studied (e.g., by Altmann, Christophersen, Ilten, etc.), but it concerns **deformations of the ring structure** — varying the coefficients, adding higher-order terms, etc. — not **varying the threshold**.

**What is true:** If you vary the threshold τ, the simplicial complex Δ_τ changes. The **persistence** of topological features across thresholds is studied by **persistent homology**, not deformation theory. The two are related (both study "variation of structure") but not identical.

**Persistent homology** is the correct tool for threshold stability. **Deformation theory** is about algebraic moduli. I conflated them.

---

## The Corrected Full Chain

```
Attention pattern (weights on complete directed graph)
        ↓
Threshold τ → Undirected graph G_τ
        ↓
Clique complex (or flag complex) Δ_τ = {cliques in G_τ}
        ↓
Stanley-Reisner ideal I_{Δ_τ} ⊂ S = k[x_1,...,x_n]
        ↓
Face ring S/I_{Δ_τ}
        ↓
Minimal free resolution → Betti table β_{i,j}
        ↓
Hochster's formula: β_{i,j} = ⊕_{|W|=j} H̃_{j-i-1}(Δ_W)
        ↓
Homology of induced subcomplexes (local-to-global)
        ↓
[Optional] Geometric realization |Δ_τ| → Homotopy type
        ↓
[Optional] Face poset P(Δ_τ) with Alexandrov topology
        ↓
[Optional] Sheaves on P(Δ_τ) → Sheaf cohomology
        ↓
[Optional] Persistent homology across varying τ
```

## The Honest Assessment

The **solid chain** is:

**Simplicial complex → Stanley-Reisner ring → Betti table → Homology of subcomplexes**

This is **provably correct** and **computationally tractable**. Everything your code does is in this chain.

The **extensions** I mentioned (homotopy, sheaves, deformation theory) are:

- **Mathematically real** but **not directly computed** by your current pipeline
- **Relevant as research directions** but **not yet tools** for attention analysis
- **Require additional choices** (stalk functor, sheaf type, deformation parameters) that are not canonical

---

## What Would Make the Chain Complete

To genuinely connect to sheaf cohomology, you'd need:

1. **Define the stalk functor**: Assign to each face σ ⊂ Δ the vector space spanned by the tokens in σ (or their embeddings, or the head's output on that subset)
2. **Define restriction maps**: Linear maps between these spaces corresponding to face inclusions
3. **Verify sheaf axioms**: Check that the gluing conditions match attention dynamics
4. **Compute sheaf cohomology**: Use the poset topology or Čech methods

This is **doable** but **not standard**. It would be genuine research, not application of existing theory.

To connect to deformation theory / threshold stability, you'd need:

1. **Persistent homology**: Track Betti numbers across varying τ
2. **Barcode analysis**: Identify which features are stable (long persistence) vs. noise (short persistence)
3. **Stability theorems**: Apply results from persistent homology (e.g., stability under perturbations)

This is **also doable** and **more standard** than sheaf theory for this application.
